# miRanda Target Scan

![miRanda Target Scan](https://proto-bio.github.io/proto-assets/images/tool/miranda/hero.png)

This notebook demonstrates `run_miranda_scan`, which predicts microRNA target sites in RNA/DNA sequences by combining Smith-Waterman complementarity alignment with ViennaRNA duplex thermodynamics. The example reproduces the canonical *Drosophila* miR-bantam / *hid* 3'UTR interaction. See the miRanda paper for background: [Enright et al., Genome Biology (2003)](https://doi.org/10.1186/gb-2003-5-1-r1).

In [1]:
from proto_tools.tools.gene_annotation.miranda import run_miranda_scan, MirandaInput, MirandaConfig
from proto_tools.utils.notebook_docs import display_api_reference

## Input

`MirandaInput` takes one or more target sequences to scan. The microRNA query set lives on `MirandaConfig` (`mirna_queries`) because the same queries apply to every target in a scan, so they're a shared configuration rather than per-target data.

In [2]:
display_api_reference("miranda-scan", "input", "run_miranda_scan")

**Input** — `MirandaInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>target_sequences</code> | <code>list[str]</code> | required | RNA/DNA target sequences (mRNA, 3'UTR, genomic) to scan for target sites |

## Config

`MirandaConfig` controls scoring thresholds and alignment penalties. Defaults mirror the miRanda CLI; lower `score_threshold` or raise `energy_threshold` toward 0 for higher sensitivity.

In [3]:
display_api_reference("miranda-scan", "config", "run_miranda_scan")

**Config** — `MirandaConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>mirna_queries</code> | <code>list[str] &#124; None</code> | <code>None</code> | microRNA query sequences to scan with; applied to every target sequence |
| <code>mirna_ids</code> | <code>list[str] &#124; None</code> | <code>None</code> | Optional microRNA identifiers surfaced on each site (defaults to seq_0, seq_1, ...) |
| <code>score_threshold</code> | <code>int</code> | <code>50</code> | Minimum alignment score to report a site; lower for higher sensitivity |
| <code>energy_threshold</code> | <code>int</code> | <code>-20</code> | Max duplex free energy (kcal/mol) to report; negative, raise toward 0 for more sensitivity |
| <code>scale</code> | <code>float</code> | <code>4.0</code> | Contrast scaling on 5' seed match/mismatch scores; higher emphasizes seed complementarity |
| <code>gap_open</code> | <code>int</code> | <code>-8</code> | Penalty for opening a gap in the alignment; more negative discourages gaps |
| <code>gap_extend</code> | <code>int</code> | <code>-2</code> | Per-position penalty for extending an open gap; more negative discourages long gaps |
| <code>strict</code> | <code>bool</code> | <code>False</code> | Strict miRNA:target 5'-seed heuristics (`-strict`); enable for fewer, higher-confidence sites |
| <code>compute_energy</code> | <code>bool</code> | <code>True</code> | Compute ViennaRNA duplex free energy; disable (`-noenergy`) for a faster score-only scan |
| <code>trim</code> | <code>int</code> | <code>0</code> | Trim target sequences to this many nucleotides (0 = no trimming) |

## Output

`MirandaOutput` returns one `MirandaSequenceResult` per target sequence, each holding the predicted `target_sites` sorted by score descending.

In [4]:
display_api_reference("miranda-scan", "output", "run_miranda_scan")

**Output** — `MirandaOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[MirandaSequenceResult]</code> | <code>[]</code> | Per-target prediction results, one per input target sequence |

**`MirandaSequenceResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>target_id</code> | <code>str</code> | required | Identifier of the target sequence |
| <code>target_sequence</code> | <code>str</code> | required | The input target sequence |
| <code>target_sites</code> | <code>list[MirandaTargetSite]</code> | <code>[]</code> | Predicted microRNA target sites in this sequence, sorted by score descending |

**`MirandaTargetSite`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>mirna_id</code> | <code>str</code> | required | Identifier of the microRNA query that hit this site |
| <code>score</code> | <code>float</code> | required | Smith-Waterman complementarity alignment score |
| <code>energy</code> | <code>float</code> | required | ViennaRNA duplex free energy (kcal/mol); 0.0 if energy disabled |
| <code>target_start</code> | <code>int</code> | required | 1-indexed inclusive start on the target sequence |
| <code>target_end</code> | <code>int</code> | required | 1-indexed inclusive end on the target sequence |
| <code>mirna_start</code> | <code>int</code> | required | 1-indexed inclusive start on the microRNA query |
| <code>mirna_end</code> | <code>int</code> | required | 1-indexed inclusive end on the microRNA query |
| <code>alignment_length</code> | <code>int</code> | required | Length of the aligned region |
| <code>identity</code> | <code>float</code> | required | Percent Watson-Crick identity over the alignment |
| <code>similarity</code> | <code>float</code> | required | Percent similarity (matches plus G:U wobble) over the alignment |
| <code>mirna_alignment</code> | <code>str</code> | <code>''</code> | microRNA strand (3'->5') |
| <code>pairing</code> | <code>str</code> | <code>''</code> | Pairing annotation (`\|` match, `:` G:U wobble) |
| <code>target_alignment</code> | <code>str</code> | <code>''</code> | Target strand (5'->3') |

## Basic usage

Scan the real *Drosophila* miR-bantam microRNA against the *hid* 3'UTR. The miR-bantam sequence is short, so we paste it inline; the *hid* 3'UTR is long, so we read it from the bundled `hid_utr.fasta` next to this notebook.

In [5]:
from pathlib import Path


def read_fasta_sequence(path):
    """Return the concatenated sequence from a single-record FASTA file."""
    lines = Path(path).read_text().splitlines()
    return "".join(line.strip() for line in lines if not line.startswith(">"))


# miR-bantam microRNA query (real sequence) and the hid 3'UTR target.
mir_bantam = "gtgagatcattttgaaagctg"
hid_utr = read_fasta_sequence("hid_utr.fasta")

inputs = MirandaInput(target_sequences=[hid_utr])
config = MirandaConfig(mirna_queries=[mir_bantam], mirna_ids=["miR-bantam"])

# Run the scan with default scoring thresholds.
result = run_miranda_scan(inputs, config)

# Results come back one per input target, in order. Inspect the hid 3'UTR hits.
hid_result = result.results[0]
print(f"hid 3'UTR: {hid_result.num_sites} target site(s) found")

top_site = hid_result.target_sites[0]
print(f"  microRNA:     {top_site.mirna_id}")
print(f"  score:        {top_site.score}")
print(f"  energy:       {top_site.energy} kcal/mol")
print(f"  target_range: {top_site.target_start}-{top_site.target_end} (1-indexed, inclusive)")

Running run_miranda_scan [00:00]

hid 3'UTR: 2 target site(s) found
  microRNA:     miR-bantam
  score:        167.0
  energy:       -24.54 kcal/mol
  target_range: 1720-1740 (1-indexed, inclusive)


## Advanced usage

Loosen the search with a non-default config. Disabling `strict` (`-loose`) drops the strict 5' seed-pairing heuristic and raising `energy_threshold` toward 0 admits weaker duplexes. This finds more candidate sites at the cost of more false positives.

In [6]:
# More sensitive: no strict 5' heuristic and a looser energy cutoff.
sensitive_config = MirandaConfig(
    mirna_queries=[mir_bantam], mirna_ids=["miR-bantam"], strict=False, energy_threshold=-14
)
sensitive_result = run_miranda_scan(inputs, sensitive_config)

default_sites = result.results[0].num_sites
sensitive_sites = sensitive_result.results[0].num_sites
print(f"default config:   {default_sites} site(s)")
print(f"sensitive config: {sensitive_sites} site(s)")
print(f"additional sites found: {sensitive_sites - default_sites}")

Running run_miranda_scan [00:00]

default config:   2 site(s)
sensitive config: 4 site(s)
additional sites found: 2


## Export results

Predicted sites can be exported to `csv` or `json`; both are one flat row per target site. Here we write the default-config results to CSV and print the table head.

In [7]:
import pandas as pd

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Supported formats: result.output_format_options -> ['csv', 'json'].
result.export(name="miranda_sites", export_path=output_dir, file_format="csv")

print(pd.read_csv(output_dir / "miranda_sites.csv").head())

  target_id    mirna_id  target_start  target_end  mirna_start  mirna_end  \
0     seq_0  miR-bantam          1720        1740            2         20   
1     seq_0  miR-bantam           885         905            2         17   

   score  energy  alignment_length  identity  similarity  
0  167.0  -24.54                18     83.33       94.44  
1  156.0  -20.03                15     86.67       93.33  
